# Utilização da extensão 'Parquet' - Transformando arquivos 'xlsx' em 'parquet': Criado pelo Nicholas Birochi

#### Importando bibliotecas essenciais para a Extração de Dados:

In [ ]:
# Importando Bibliotecas
import numpy as np
import pandas as pd
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, date

# -----------------------------
# Parâmetros do seu padrão (mantidos, mas sem aplicar o corte posicional)
# -----------------------------
KEEP_N = 67
DROP_N = 5
CYCLE = KEEP_N + DROP_N

# -----------------------------
# Leitura robusta do XLSX
# -----------------------------
def _find_soffice() -> str | None:
    candidates = [
        r"C:\Program Files\LibreOffice\program\soffice.exe",
        r"C:\Program Files (x86)\LibreOffice\program\soffice.exe",
        "soffice",
    ]
    for c in candidates:
        try:
            p = subprocess.run([c, "--version"], capture_output=True, text=True)
            if p.returncode == 0:
                return c
        except Exception:
            pass
    return None

def _convert_with_libreoffice_to_xlsx(path: Path) -> Path:
    soffice = _find_soffice()
    if not soffice:
        raise RuntimeError(
            "Falha ao ler esse XLSX (formatação corrompida) e não achei o LibreOffice (soffice).\n"
            "Instale o LibreOffice OU re-salve o arquivo no Excel/LibreOffice (Salvar como .xlsx) e rode de novo."
        )

    tmpdir = Path(tempfile.mkdtemp(prefix="xlsx_clean_"))
    cmd = [
        soffice, "--headless", "--nologo", "--nolockcheck", "--norestore",
        "--convert-to", "xlsx",
        "--outdir", str(tmpdir),
        str(path),
    ]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Falha ao converter com LibreOffice.\nSTDERR:\n{p.stderr}\nSTDOUT:\n{p.stdout}")

    converted = tmpdir / f"{path.stem}.xlsx"
    if not converted.exists():
        raise RuntimeError("LibreOffice não gerou o arquivo convertido esperado.")
    return converted

def read_excel_robusto(path: Path) -> pd.DataFrame:
    # 1) tenta openpyxl
    try:
        return pd.read_excel(path, header=None, engine="openpyxl")
    except Exception:
        pass

    # 2) tenta python-calamine (se instalado)
    try:
        from python_calamine import CalamineWorkbook
        wb = CalamineWorkbook.from_path(str(path))
        sheet = wb.sheet_names[0]
        rows = wb.get_sheet_by_name(sheet).to_python(skip_empty_area=False)
        return pd.DataFrame(rows)
    except Exception:
        pass

    # 3) converte com LibreOffice e lê de novo
    converted = _convert_with_libreoffice_to_xlsx(path)
    return pd.read_excel(converted, header=None, engine="openpyxl")

# -----------------------------
# Helpers: hora / escolha de colunas por posição + fallback
# Ordem informada (após remover 3 linhas + pular header):
# Cadastro, Nome, NADA, Filial, Data, Hora, Dir, Tipo Acesso, Uso Acesso, Planta, Coletor, Descricao Coletor, 1
# -----------------------------
def _clean_time_series(s: pd.Series) -> pd.Series:
    raw = s.copy()

    # ---------- caminho A: numérico (serial do Excel) ----------
    num = pd.to_numeric(raw, errors="coerce")
    frac = (num % 1)
    is_time_num = num.notna() & (frac != 0)

    secs = (frac.where(is_time_num) * 86400).round()
    td = pd.to_timedelta(secs, unit="s", errors="coerce")

    def _fmt_td(x):
        if pd.isna(x):
            return pd.NA
        total = int(x.total_seconds())
        hh = total // 3600
        mm = (total % 3600) // 60
        ss = total % 60
        return f"{hh:02d}:{mm:02d}:{ss:02d}"

    out_num = td.map(_fmt_td).astype("string")

    # ---------- caminho B: texto ----------
    st = raw.astype("string").str.strip()
    st = st.str.replace(r"[\[\]]", "", regex=True)
    st = st.str.replace("\u00a0", "", regex=False)
    st = st.str.replace(r"\s+", "", regex=True)
    st = st.mask(st.isna() | st.str.fullmatch(r""), pd.NA)

    st = st.mask(
        st.str.fullmatch(r"0+|00:00|00:00:00|0:00|0:00:00", na=False),
        pd.NA
    )

    st = st.str.replace(r"^(\d{1,2}):(\d{2})$", r"\1:\2:00", regex=True)

    # ALTERAÇÃO NECESSÁRIA: parse por elemento para aceitar formatos mistos
    def _parse_hora_txt(x):
        if pd.isna(x):
            return pd.NA
        try:
            dt = pd.to_datetime(f"1970-01-01 {x}", errors="coerce")
            if pd.isna(dt):
                return pd.NA
            return dt.strftime("%H:%M:%S")
        except Exception:
            return pd.NA

    out_txt = st.map(_parse_hora_txt).astype("string")

    out = out_txt.where(~is_time_num, out_num)
    out = out.mask(out.isin(["00:00:00", "00:00", "0:00:00", "0:00"]), pd.NA)

    return out


def _int_rate(col: pd.Series) -> float:
    s = col.astype("string").str.strip().fillna("")
    m = s.str.fullmatch(r"\d+")
    return float(m.mean()) if len(m) else 0.0

def _pick_filial_idx(df: pd.DataFrame) -> int:
    candidates = [i for i in range(2, min(5, df.shape[1]))]
    if not candidates:
        return 3
    rates = [(i, _int_rate(df.iloc[:, i])) for i in candidates]
    rates.sort(key=lambda x: x[1], reverse=True)
    return rates[0][0]

def _pick_desc_col(df: pd.DataFrame, start_col: int) -> int:
    start_col = max(0, min(start_col, df.shape[1] - 1))
    cols = range(start_col, df.shape[1])
    best_idx, best_score = start_col, -1e9
    sample = df.head(300)

    for i in cols:
        s = sample.iloc[:, i].astype("string").str.strip()
        has_letters = s.str.contains(r"[A-Za-zÀ-ÿ]", na=False).mean()
        has_dash = s.str.contains(r"\s-\s", na=False).mean()
        all_dash = (s == "-").mean()
        score = has_letters + has_dash - (2.0 * all_dash)
        if score > best_score:
            best_idx, best_score = i, score

    return best_idx

def _parse_data_series(raw: pd.Series) -> pd.Series:
    out = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")

    # 1) valores que já vêm como data/datetime do Excel/openpyxl
    mask_dtlike = raw.map(
        lambda x: isinstance(x, (pd.Timestamp, datetime, date, np.datetime64))
    )
    if mask_dtlike.any():
        out.loc[mask_dtlike] = pd.to_datetime(
            raw.loc[mask_dtlike],
            errors="coerce",
            dayfirst=True,
        )

    # 2) strings
    s = raw.astype("string").str.strip()

    # 2a) dd/mm/yyyy
    mask_slash_date = s.str.fullmatch(r"\d{2}/\d{2}/\d{4}", na=False) & out.isna()
    if mask_slash_date.any():
        out.loc[mask_slash_date] = pd.to_datetime(
            s.loc[mask_slash_date],
            format="%d/%m/%Y",
            errors="coerce",
        )

    # 2b) dd/mm/yyyy hh:mm ou hh:mm:ss
    mask_slash_dt = s.str.fullmatch(r"\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}(:\d{2})?", na=False) & out.isna()
    if mask_slash_dt.any():
        vals = s.loc[mask_slash_dt].str.replace(r"\s+", " ", regex=True)

        parsed1 = pd.to_datetime(
            vals,
            format="%d/%m/%Y %H:%M:%S",
            errors="coerce",
        )
        out.loc[mask_slash_dt] = parsed1

        ainda = mask_slash_dt & out.isna()
        if ainda.any():
            vals2 = s.loc[ainda].str.replace(r"\s+", " ", regex=True)
            out.loc[ainda] = pd.to_datetime(
                vals2,
                format="%d/%m/%Y %H:%M",
                errors="coerce",
            )

    # 2c) ISO yyyy-mm-dd ou yyyy-mm-dd hh:mm:ss
    mask_iso = s.str.match(r"^\d{4}-\d{2}-\d{2}([ T]\d{2}:\d{2}(:\d{2})?)?$", na=False) & out.isna()
    if mask_iso.any():
        out.loc[mask_iso] = pd.to_datetime(
            s.loc[mask_iso],
            errors="coerce",
            dayfirst=False,
        )

    # 3) numérico excel serial
    num = pd.to_numeric(raw, errors="coerce")
    mask_num = num.notna() & out.isna()
    if mask_num.any():
        out.loc[mask_num] = pd.to_datetime(
            num.loc[mask_num],
            unit="d",
            origin="1899-12-30",
            errors="coerce",
        )

    # 4) fallback final: SEMPRE brasileiro
    return out

def _round_hora_minuto(h: pd.Series) -> pd.Series:
    st = h.astype("string")

    # ALTERAÇÃO NECESSÁRIA: parse por elemento para aceitar formatos mistos
    def _round_one(x):
        if pd.isna(x):
            return pd.NA
        try:
            dt = pd.to_datetime(f"1970-01-01 {x}", errors="coerce")
            if pd.isna(dt):
                return pd.NA
            dt = dt.round("min")
            return dt.strftime("%H:%M:%S")
        except Exception:
            return pd.NA

    return st.map(_round_one).astype("string")

# -----------------------------
# Limpeza + manter só: cadastro, nome, data, hora, dir, planta, descricao coletor
# -----------------------------
def load_and_clean_excel(path: Path) -> pd.DataFrame:
    df = read_excel_robusto(path)

    # remove 3 primeiras linhas
    df = df.iloc[3:].reset_index(drop=True)

    # pula a linha do cabeçalho do xlsx
    df = df.iloc[1:].reset_index(drop=True)

    # preserva "NADA" (não dropa colunas intermediárias)
    df = df.replace(r"^\s*$", np.nan, regex=True)

    # corta no último campo não-nulo
    any_nonnull = df.notna().any(axis=0).to_numpy()
    if any_nonnull.any():
        last = int(np.where(any_nonnull)[0].max())
        df = df.iloc[:, : last + 1]

    # índices fixos + filial detectada
    cadastro_idx = 0
    nome_idx = 1
    filial_idx = _pick_filial_idx(df)

    data_idx = filial_idx + 1
    hora_idx = filial_idx + 2
    dir_idx = filial_idx + 3

    # planta fica depois de tipo acesso + uso acesso
    planta_idx = filial_idx + 6

    # descrição coletor costuma estar depois de coletor
    desc_start = filial_idx + 8
    desc_idx = _pick_desc_col(df, desc_start)

    out = pd.DataFrame({
        "cadastro": df.iloc[:, cadastro_idx],
        "nome": df.iloc[:, nome_idx],
        "data": df.iloc[:, data_idx],
        "hora": df.iloc[:, hora_idx],
        "dir": df.iloc[:, dir_idx],
        "planta": df.iloc[:, planta_idx],
        "descricao coletor": df.iloc[:, desc_idx],
    })

    out["cadastro"] = (
        out["cadastro"].astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.strip()
        .replace({"": pd.NA})
    )
    out["nome"] = out["nome"].astype("string").str.strip()
    out["descricao coletor"] = out["descricao coletor"].astype("string").str.strip()
    out["planta"] = out["planta"].astype("string").str.strip()

    # dir -> E/S
    out["dir"] = out["dir"].astype("string").str.strip().str.upper()
    out["dir"] = out["dir"].where(out["dir"].isin(["E", "S"]), pd.NA)

    # parse data
    dt_data = _parse_data_series(out["data"])

    time_from_data = dt_data.dt.strftime("%H:%M:%S").astype("string")
    time_from_data = time_from_data.where(
        (dt_data.dt.hour != 0) | (dt_data.dt.minute != 0) | (dt_data.dt.second != 0),
        pd.NA
    )

    out["data"] = dt_data.dt.strftime("%d/%m/%Y").astype("string")

    # limpa hora e preenche faltantes com hora embutida na data
    out["hora"] = _clean_time_series(out["hora"])
    out["hora"] = out["hora"].fillna(time_from_data)

    # fallback quando hora ainda está faltando ou virou 00:00:00
    missing = out["hora"].isna() | (out["hora"] == "00:00:00")

    if missing.any():
        lo = max(0, hora_idx - 3)
        hi = min(df.shape[1], hora_idx + 12)
        win = df.iloc[:, lo:hi].astype("string").loc[missing]

        pattern = r"(\[?\d{1,2}\]?:\d{2}(?::\d{2})?)"
        cands = [win.iloc[:, j].str.extract(pattern, expand=False) for j in range(win.shape[1])]
        first = pd.concat(cands, axis=1).bfill(axis=1).iloc[:, 0]

        out.loc[missing, "hora"] = _clean_time_series(first)

    # limpa hora e preenche faltantes com hora embutida na data
    out["hora"] = _clean_time_series(out["hora"])
    out["hora"] = out["hora"].fillna(time_from_data)

    # fallback 1: procura um HH:MM nas colunas vizinhas da "hora"
    missing = out["hora"].isna() | (out["hora"] == "00:00:00")

    if missing.any():
        lo = max(0, hora_idx - 3)
        hi = min(df.shape[1], hora_idx + 12)
        win = df.iloc[:, lo:hi].astype("string").loc[missing]

        pattern = r"(\[?\d{1,2}\]?:\d{2}(?::\d{2})?)"
        cands = [win.iloc[:, j].str.extract(pattern, expand=False) for j in range(win.shape[1])]
        first = pd.concat(cands, axis=1).bfill(axis=1).iloc[:, 0]

        out.loc[missing, "hora"] = _clean_time_series(first)

    # NOVO fallback 2: procura a primeira hora válida na linha inteira
    missing2 = out["hora"].isna() | (out["hora"] == "00:00:00")

    if missing2.any():
        full = df.astype("string").loc[missing2]

        pattern = r"(\[?\d{1,2}\]?:\d{2}(?::\d{2})?)"
        cands_full = [full.iloc[:, j].str.extract(pattern, expand=False) for j in range(full.shape[1])]
        first_full = pd.concat(cands_full, axis=1).bfill(axis=1).iloc[:, 0]

        out.loc[missing2, "hora"] = _clean_time_series(first_full)

    # arredonda para minuto
    out["hora"] = _round_hora_minuto(out["hora"])

    # mantém só linhas que parecem batidas válidas
    out = out[
        out["cadastro"].notna()
        & out["nome"].notna()
        & out["data"].notna()
        & out["dir"].isin(["E", "S"])
    ].copy()

    out = out.dropna(how="all").reset_index(drop=True)
    return out

# -----------------------------
# Execução: 2 XLSX -> 2 PARQUET + df_ronda.parquet
# -----------------------------
xlsx_dir = Path("xlsx")
f1 = xlsx_dir / "SPAC903_82_Colab_Tathi_Complemento_Original_.xlsx"
f2 = xlsx_dir / "SPAC903_82_Colab_Tathi_Original.xlsx"

for f in (f1, f2):
    if not f.exists():
        raise FileNotFoundError(f"Não encontrei: {f}")

df1 = load_and_clean_excel(f1)
df2 = load_and_clean_excel(f2)

df_ronda = pd.concat([df1, df2], ignore_index=True)
df_ronda.to_parquet("df_ronda.parquet", index=False)

print("OK")
print("df1:", df1.shape, "df2:", df2.shape, "df_ronda:", df_ronda.shape)
